## Cruce de controles ISOLUCION y mapas de riesgos

In [58]:
from utils import limpiar_texto, exportar_csv
import pandas as pd
import warnings
from rapidfuzz import fuzz
warnings.filterwarnings('ignore')

#### Cargar los datos del procesos de datos del primer iypnb

In [59]:
# Traer archivos generados en 01_cruce_riesgos.ipynb
df_match = pd.read_csv('df_match.csv', encoding='utf-8-sig', sep='|')
mapa_ini = pd.read_csv('mapa_ini.csv', encoding='utf-8-sig', sep='|')
mapa_rep = pd.read_csv('mapa_rep.csv', encoding='utf-8-sig', sep='|')

#### Generar la data de controles del mapa de riesgos consolidado

In [60]:
# Fuentes originales - mapa consolidado de riesgos
mapa_excel = pd.read_excel('Consolidado_2025_v13.xlsx',
                            sheet_name='Mapa Riesgos',
                            header=None)

Limpieza de datos de controles del mapa de riesgos

In [61]:
# Eliminar primeras filas (encabezado del formato)
mapa_controles = mapa_excel.iloc[10:, [0, 16, 17, 21, 22, 23, 24, 25, 26, 27]].reset_index(drop=True)
# Renombrar columnas
mapa_controles.columns = [
    'consecutivo', 'no_control', 'descripcion_control', 'afectacion',
    'tipo_control', 'implementacion_control', 'calificacion_control',
    'documentacion_control', 'frecuencia_control', 'evidencia_control'
]
# Eliminar filas innecesarias
mapa_controles = mapa_controles.iloc[2:, :].reset_index(drop=True)
# Eliminar filas en blanco
mapa_controles.dropna(subset=['tipo_control'], inplace=True)
# Reiniciar índices del df
mapa_controles.reset_index(drop=True, inplace=True)
# Rellenar vacios 
mapa_controles['consecutivo'] = mapa_controles['consecutivo'].ffill()
# Crear una llave de identificación de cada control del mapa de riesgos
mapa_controles['id_control'] = mapa_controles['consecutivo'].astype(str) + '-' + mapa_controles['no_control'].astype(str)

#### Generar la data de controles reportados a ISOLUCION

In [62]:
# Fuentes originales - isolucion
mapa_iso   = pd.read_excel('Reporte Mejoramiento Continuo - Controles.xlsx', 
                            sheet_name='Data')

In [63]:
# Seleccionar las columnas con la información de los controles y su calificación
iso_controles = mapa_iso[['Num', 'Actividad', 'Unnamed: 16']].copy()
# Renombrar columnas
iso_controles.columns = ['num', 'actividad', 'eficacia']
# Eliminar filas innecesarias y resetear indices
iso_controles = iso_controles.iloc[1:, :].reset_index(drop=True)
# Eliminar filas en blanco
iso_controles.dropna(how='all', inplace=True)
# Rellenar hacia abajo los números pendientes
iso_controles['num'] = iso_controles['num'].ffill()
# Resetear índices del DF
iso_controles.reset_index(drop=True, inplace=True)
# Crear una llave de identificación de cada control cargado en ISOLUCION
iso_controles['no_control'] = iso_controles.groupby('num').cumcount() + 1
iso_controles['id_control'] = iso_controles['num'].astype(int).astype(str) + '-' + iso_controles['no_control'].astype(str)

In [64]:
# Limpiar texto para hacer fuzzy match
mapa_controles['desc_limpia'] = mapa_controles['descripcion_control'].apply(limpiar_texto)
iso_controles['actividad_limpia'] = iso_controles['actividad'].apply(limpiar_texto)

### Cruce de datas de controles

In [65]:
# Eliminar controles duplicados confirmados en Isolucion
iso_controles = iso_controles[iso_controles['id_control'] != '5431-2'].reset_index(drop=True)

In [66]:
# Para cada fila de iso_controles, tomar su num
# Buscar en df_match el consecutivo_ini correspondiente a ese num
# Filtrar mapa_controles por ese consecutivo_ini
# Hacer fuzzy match de actividad_limpia vs desc_limpia dentro de ese subconjunto

resultados = []

for _, iso_row in iso_controles.iterrows():
    num = iso_row['num']
    
    match_riesgo = df_match[df_match['num_rep'] == num]
    if len(match_riesgo) == 0:
        continue
    consecutivo = match_riesgo.iloc[0]['consecutivo_ini']
    
    candidatos = mapa_controles[mapa_controles['consecutivo'] == consecutivo]
    if len(candidatos) == 0:
        continue
    
    mejor_score = 0
    mejor_control = None
    for _, ctrl_row in candidatos.iterrows():
        score = fuzz.token_sort_ratio(iso_row['actividad_limpia'], ctrl_row['desc_limpia'])
        if score > mejor_score:
            mejor_score = score
            mejor_control = ctrl_row
    
    resultados.append({
        'num': num,
        'id_control_iso': iso_row['id_control'],
        'consecutivo': consecutivo,
        'id_control_mapa': mejor_control['id_control'],
        'no_control_match': mejor_control['no_control'],
        'score_control': mejor_score,
        'eficacia': iso_row['eficacia']
    })

df_controles = pd.DataFrame(resultados)

In [67]:
# Listado de cruce de controles mapa vs ISOLUCION con un valor match mabo
bajos = df_controles[df_controles['score_control'] < 70][['num', 'id_control_iso', 'id_control_mapa', 'score_control', 'eficacia']]
print(bajos.shape)
print(bajos)

(14, 5)
        num id_control_iso id_control_mapa  score_control eficacia
72   5473.0         5473-1             1-2      60.484429       SI
74   5474.0         5474-2             2-2      63.399694       NO
97   5441.0         5441-1            86-2      60.184409       SI
115  5423.0         5423-2            73-2      68.869386       SI
162  5411.0         5411-1            44-1      67.848101       SI
163  5411.0         5411-2            44-2      69.278607       SI
164  5412.0         5412-1            45-1      66.896949       NO
165  5412.0         5412-2            45-2      56.142301       NO
168  5415.0         5415-1            98-1      57.868020       SI
169  5415.0         5415-2            98-3      55.267423       SI
170  5415.0         5415-3            98-3      60.849057       SI
175  5418.0         5418-1            99-1      64.379947       SI
176  5418.0         5418-2            99-4      49.910233       SI
177  5418.0         5418-3            99-4      55.230

In [68]:
# Controles por consecutivo en mapa
conteo_mapa = mapa_controles.groupby('consecutivo').size().reset_index(name='n_controles_mapa')

# Controles por consecutivo en iso (via df_match)
conteo_iso = df_controles.groupby('consecutivo').size().reset_index(name='n_controles_iso')

# Unir con df_match para traer el num
comparacion = conteo_mapa.merge(
    df_match[['consecutivo_ini', 'num_rep']].rename(columns={'consecutivo_ini':'consecutivo', 'num_rep':'num'}),
    on='consecutivo', how='left'
).merge(conteo_iso, on='consecutivo', how='left')

comparacion['n_controles_iso'] = comparacion['n_controles_iso'].fillna(0).astype(int)

In [69]:
diferencias = comparacion[comparacion['n_controles_mapa'] != comparacion['n_controles_iso']]
print(diferencias.to_string())

    consecutivo  n_controles_mapa     num  n_controles_iso
0             1                 2  5473.0                1
17           18                 1     NaN                0
18           19                 3  5438.0                9
54           55                 1  5403.0                3
98           99                 4  5418.0                3


In [70]:

# Reconstruir tablas diff
ctrl_mapa_diff = mapa_controles[mapa_controles['consecutivo'].isin(diferencias['consecutivo'])][
    ['consecutivo', 'id_control', 'descripcion_control', 'desc_limpia']
].copy()

ctrl_iso_diff = iso_controles[iso_controles['num'].isin(diferencias['num'])][
    ['num', 'id_control', 'actividad', 'actividad_limpia']
].copy()

# Traer consecutivo a iso_diff via diferencias
ctrl_iso_diff = ctrl_iso_diff.merge(
    diferencias[['consecutivo', 'num']],
    on='num', how='left'
)

# Fuzzy match dentro de cada consecutivo
resultados_diff = []
for consecutivo, grupo_mapa in ctrl_mapa_diff.groupby('consecutivo'):
    grupo_iso = ctrl_iso_diff[ctrl_iso_diff['consecutivo'] == consecutivo]
    for _, iso_row in grupo_iso.iterrows():
        for _, mapa_row in grupo_mapa.iterrows():
            score = fuzz.token_sort_ratio(iso_row['actividad_limpia'], mapa_row['desc_limpia'])
            resultados_diff.append({
                'consecutivo': consecutivo,
                'id_control_mapa': mapa_row['id_control'],
                'id_control_iso': iso_row['id_control'],
                'score': score
            })

df_diff_scores = pd.DataFrame(resultados_diff)

In [71]:
# Para cada id_control_iso tomar el mejor score y asignar los que se pueda de manera correcta
df_diff_scores = df_diff_scores.sort_values('score', ascending=False)

asignaciones = []
mapa_usados = {}

for consecutivo, grupo in df_diff_scores.groupby('consecutivo'):
    usados = set()
    for _, row in grupo.iterrows():
        if row['id_control_mapa'] not in usados:
            asignaciones.append(row)
            usados.add(row['id_control_mapa'])

df_asignaciones_diff = pd.DataFrame(asignaciones)
print(df_asignaciones_diff.shape)
print(df_asignaciones_diff.sort_values('consecutivo').to_string())

(10, 4)
    consecutivo id_control_mapa id_control_iso      score
1             1             1-2         5473-1  60.484429
0             1             1-1         5473-1  39.041361
2            19            19-1         5438-1  97.377622
12           19            19-2         5438-4  94.645441
22           19            19-3         5438-7  91.014015
31           55            55-1         5403-3  97.495528
32           99            99-1         5418-1  64.379947
43           99            99-4         5418-3  55.230126
37           99            99-2         5418-2  44.779116
34           99            99-3         5418-1  43.048404


Una vez revisados los controles de manera manual: cruzando mapa e isolución, se hace el ajuste manual a la data

In [72]:
# Correcciones manuales: id_control_iso → id_control_mapa correcto
correcciones = {
    # Consecutivo 19 / 5438 — réplicas por secretaría
    '5438-2': '19-1',
    '5438-3': '19-1',
    '5438-5': '19-2',
    '5438-6': '19-2',
    '5438-8': '19-3',
    '5438-9': '19-3',
    # Consecutivo 55 / 5403 — réplicas por secretaría
    '5403-1': '55-1',
    '5403-2': '55-1',
    # Consecutivo 86 / 5441
    '5441-1': '86-1',
    '5441-2': '86-2',
    # Consecutivo 80 / 5430
    '5430-3': '80-3',
    # Consecutivo 98 / 5415
    '5415-2': '98-2',
    '5415-3': '98-3',
    # Consecutivo 99 / 5418
    '5418-1': '99-1',
    '5418-2': '99-2',
    '5418-3': '99-3',
}

# Aplicar correcciones sobre df_controles
for id_iso, id_mapa_correcto in correcciones.items():
    df_controles.loc[df_controles['id_control_iso'] == id_iso, 'id_control_mapa'] = id_mapa_correcto

# Anotación: 1-1 sin par en Isolucion (control no cargado)
sin_par_mapa = ['1-1']

print("Correcciones aplicadas:", len(correcciones))
print("Controles sin par en Isolucion:", sin_par_mapa)

Correcciones aplicadas: 16
Controles sin par en Isolucion: ['1-1']


Frente a los controles que no fueron cargados en ISOLUCION, se hace el ajuste manual

Una nota: la columna motivo se crea para anotar el ajuste manual. Las filas anteriores quedarán con NaN en esa columna, lo cual es correcto porque solo aplica para estos dos casos especiales: 1-1 y 18-1 (numeración del consolidado 2025 mapa de riesgos)

In [73]:
# Controles sin seguimiento en Isolucion
sin_seguimiento = pd.DataFrame([
    {
        'num': 5473.0,
        'id_control_iso': 'SIN_CARGA',
        'consecutivo': 1,
        'id_control_mapa': '1-1',
        'no_control_match': 1,
        'score_control': 0,
        'eficacia': 'SIN_SEGUIMIENTO',
        'motivo': 'Control no registrado en Isolucion — incumplimiento metodológico vigencia anterior'
    },
    {
        'num': None,
        'id_control_iso': 'SIN_CARGA',
        'consecutivo': 18,
        'id_control_mapa': '18-1',
        'no_control_match': 1,
        'score_control': 0,
        'eficacia': 'SIN_SEGUIMIENTO',
        'motivo': 'Riesgo no registrado en Isolucion — incumplimiento metodológico vigencia anterior'
    },
    {
        'num': None,
        'id_control_iso': 'SIN_CARGA',
        'consecutivo': 99,
        'id_control_mapa': '99-4',
        'no_control_match': 4,
        'score_control': 0,
        'eficacia': 'SIN_SEGUIMIENTO',
        'motivo': 'Control no registrado en Isolucion — incumplimiento metodológico vigencia anterior'
    }
])

df_controles = pd.concat([df_controles, sin_seguimiento], ignore_index=True)

print(df_controles[df_controles['eficacia'] == 'SIN_SEGUIMIENTO'][
    ['consecutivo', 'id_control_mapa', 'id_control_iso', 'eficacia', 'motivo']
])

     consecutivo id_control_mapa id_control_iso         eficacia  \
232            1             1-1      SIN_CARGA  SIN_SEGUIMIENTO   
233           18            18-1      SIN_CARGA  SIN_SEGUIMIENTO   
234           99            99-4      SIN_CARGA  SIN_SEGUIMIENTO   

                                                motivo  
232  Control no registrado en Isolucion — incumplim...  
233  Riesgo no registrado en Isolucion — incumplimi...  
234  Control no registrado en Isolucion — incumplim...  


Resumen del ejercicio de cruce realizado para determinar eficacia por control y asignar al mapa de riesgos en Excel

In [74]:
print(df_controles['eficacia'].value_counts())
print(f"\nTotal controles: {len(df_controles)}")

eficacia
SI                 196
NO                  36
SIN_SEGUIMIENTO      3
Name: count, dtype: int64

Total controles: 235


Crear la tabla de dimensión con el dato de los controles para el modelo de datos

In [75]:
mapa_controles.columns

Index(['consecutivo', 'no_control', 'descripcion_control', 'afectacion',
       'tipo_control', 'implementacion_control', 'calificacion_control',
       'documentacion_control', 'frecuencia_control', 'evidencia_control',
       'id_control', 'desc_limpia'],
      dtype='object')

In [76]:
# Crear la tabla de la dimensión con las columnas de interés
dim_control = mapa_controles[[
    'consecutivo', 'id_control', 'no_control',
    'descripcion_control', 'tipo_control',
    'implementacion_control',
    'frecuencia_control', 'evidencia_control'
]].copy()

In [77]:
dim_control = dim_control.merge(
    df_controles[['id_control_mapa', 'id_control_iso','num']].drop_duplicates(),
    left_on='id_control',
    right_on='id_control_mapa',
    how='left'
).drop(columns='id_control_mapa')

Crear la tablas de hechos y dimensiones que contenga los resultados de la eficacia por control y los datos de los controles

In [78]:
exportar_csv(dim_control, 'dim_control.csv')
exportar_csv(df_controles, 'fact_eficacia.csv')
exportar_csv(iso_controles, 'iso_controles.csv')

print(f"dim_control: {dim_control.shape}")
print(f"fact_eficacia: {df_controles.shape}")
print(f"iso_controles: {iso_controles.shape}")

dim_control: (235, 10)
fact_eficacia: (235, 8)
iso_controles: (233, 6)


# Pruebas

In [80]:
sin_par = dim_control[~dim_control['id_control'].isin(df_controles['id_control_mapa'])]
print(sin_par[['consecutivo', 'id_control']].to_string())
print(f"\ndim_control: {len(dim_control)}")
print(f"df_controles: {len(df_controles)}")

Empty DataFrame
Columns: [consecutivo, id_control]
Index: []

dim_control: 235
df_controles: 235
